In [29]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from glob import glob
from skimage.io import imread
from sklearn.model_selection import train_test_split
import torch
from PIL import Image

In [22]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Down part
        for feat in features:
            self.downs.append(DoubleConv(in_channels, feat))
            in_channels = feat

        # Up part
        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feat*2, feat, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feat*2, feat))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            up_transpose = self.ups[idx]
            up_double = self.ups[idx+1]
            x = up_transpose(x)
            skip = skip_connections[idx//2]
            # pad if needed
            if x.shape != skip.shape:
                x = TF.resize(x, skip.shape[2:])
            x = torch.cat((skip, x), dim=1)
            x = up_double(x)

        return torch.sigmoid(self.final_conv(x))


In [3]:
def load_data(img_paths, mask_paths):
    X, Y = [], []
    for img_p, mask_p in zip(img_paths, mask_paths):
        img = imread(img_p)
        mask = imread(mask_p)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT))
        if img.ndim == 2:
            img = np.expand_dims(img, -1)
        mask = (mask > 0).astype(np.float32)
        X.append(img)
        Y.append(np.expand_dims(mask, -1))
    X = np.array(X, dtype=np.float32) / 255.0
    Y = np.array(Y, dtype=np.float32)
    return X, Y

In [13]:
data_path = "/net/birdstore/Active_Atlas_Data/data_root/brains_info/masks/structures/TG"
image_dir = os.path.join(data_path, "thumbnail_aligned")
mask_dir = os.path.join(data_path, "thumbnail_masked")
image_paths = sorted(glob(os.path.join(image_dir, "*.tif")))
mask_paths = sorted(glob(os.path.join(mask_dir, "*.tif")))

print(f"Found {len(image_paths)} images and {len(mask_paths)} masks.")

# Load and preprocess (resize to uniform shape)
IMG_HEIGHT, IMG_WIDTH = 256, 256

def load_data(img_paths, mask_paths):
    X, Y = [], []
    for img_p, mask_p in zip(img_paths, mask_paths):
        img = imread(img_p)
        mask = imread(mask_p)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT))
        if img.ndim == 2:
            img = np.expand_dims(img, -1)
        mask = (mask > 0).astype(np.float32)
        X.append(img)
        Y.append(np.expand_dims(mask, -1))
    X = np.array(X, dtype=np.float32) / 255.0
    Y = np.array(Y, dtype=np.float32)
    return X, Y

def load_model_from_checkpoint(path, device):
    ckpt = torch.load(path, map_location=device)
    features = ckpt.get('features', [32,64,128])
    in_ch = ckpt.get('in_channels', 3)
    out_ch = ckpt.get('out_channels', 1)
    model = UNet(in_channels=in_ch, out_channels=out_ch, features=features).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


X, Y = load_data(image_paths, mask_paths)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

Found 560 images and 560 masks.


In [5]:
print("Train shape:", X_train.shape, Y_train.shape)
print("Validation shape:", X_val.shape, Y_val.shape)

Train shape: (448, 256, 256, 1) (448, 256, 256, 1)
Validation shape: (112, 256, 256, 1) (112, 256, 256, 1)


In [49]:
if torch.cuda.is_available(): 
    device = torch.device('cuda') 
    print('Using Nvidia graphics card GPU.')
else:
    device = torch.device('cpu')
    print('No Nvidia card found, using CPU.')
model_path = os.path.join(data_path, "models", "best_unet.pth")

if os.path.exists(model_path):    
    model = load_model_from_checkpoint(model_path, device)    
    print(f'loaded model: {model_path}')
else:
    print(f'model not found at {model_path}')


No Nvidia card found, using CPU.


KeyError: 'model_state_dict'

In [47]:
def predict_image(model, device, img_np, resize=None):
    # img_np: HxWxC float [0..1]
    model.eval()
    img = img_np.copy()
    #if resize is not None:
    #    img = cv2.resize(img, tuple(resize[::-1]))
    #tensor = torch.from_numpy(np.transpose(img, (2,0,1))).float().unsqueeze(0).to(device)
    tensor = torch.from_numpy(img).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(tensor)
        pred_np = pred.squeeze().cpu().numpy() # [H,W]
    return pred_np

def extract_contours_from_mask(bin_mask, min_area=10):
    # bin_mask expected binary 0/1 or 0/255, uint8
    if bin_mask.dtype != np.uint8:
        bin_mask = (bin_mask > 0).astype(np.uint8) * 255
    contours, hierarchy = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filtered = []
    for c in contours:
        area = cv2.contourArea(c)
        if area >= min_area:
            filtered.append(c)
    return filtered


In [48]:
# Example usage:
from tifffile import imread
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model_from_checkpoint(model_path, device)
img_path = os.path.join(data_path, 'thumbnail_aligned/DK78.38.60.158.tif')
img = imread(img_path)
print(img.shape)
pred_mask = predict_image(model, device, img, resize=img.shape)
bin_mask = (pred_mask > 0.5).astype(np.uint8) * 255
contours = extract_contours_from_mask(bin_mask, min_area=20)
# # draw contours on original (resized) image
disp = (cv2.resize((img*255).astype(np.uint8), bin_mask.shape[::-1]))
cv2.drawContours(disp, contours, -1, (0,255,0), 2)
plt.imshow(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB))
plt.axis('off')

(1326, 2244)


RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[1, 1, 1326, 2244] to have 3 channels, but got 1 channels instead